In [93]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
print("Libraries loaded")

Libraries loaded


In [94]:
# data klasöründeki parquet dosyalarını alıyorum
files = glob.glob("../data/*.parquet")
print("Number of files:", len(files))
files[:3]

Number of files: 12


['../data/yellow_tripdata_2015-10.parquet',
 '../data/yellow_tripdata_2015-09.parquet',
 '../data/yellow_tripdata_2015-08.parquet']

In [95]:
# İlk olarak sadece Ocak ayı verisini inceliyorum
df_january = pd.read_parquet("../data/yellow_tripdata_2015-01.parquet")
df_january.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2015-01-01 00:11:33,2015-01-01 00:16:48,1,1.0,1,N,41,166,1,5.7,0.5,0.5,1.40,0.0,0.0,8.40,None,None
1,1,2015-01-01 00:18:24,2015-01-01 00:24:20,1,0.9,1,N,166,238,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None
2,1,2015-01-01 00:26:19,2015-01-01 00:41:06,1,3.5,1,N,238,162,1,13.2,0.5,0.5,2.90,0.0,0.0,17.40,None,None
3,1,2015-01-01 00:45:26,2015-01-01 00:53:20,1,2.1,1,N,162,263,1,8.2,0.5,0.5,2.37,0.0,0.0,11.87,None,None
4,1,2015-01-01 00:59:21,2015-01-01 01:05:24,1,1.0,1,N,236,141,3,6.0,0.5,0.5,0.00,0.0,0.0,7.30,None,None


# Veriye genel bakış


In [96]:
# satır ve sütun sayısının kontrol edilmesi
df_january.shape

(12741035, 19)

In [97]:
# sütun isimlerini kontrol ediyorum
df_january.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee'],
      dtype='str')

In [98]:
# veri tiplerinin kontrol edilmesi
df_january.dtypes

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   int64
trip_distance                   float64
RatecodeID                        int64
store_and_fwd_flag                  str
PULocationID                      int64
DOLocationID                      int64
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge             object
airport_fee                      object
dtype: object

In [99]:
# Eksik değerleri kontrol ediyorum
df_january.isnull().sum()

VendorID                        0
tpep_pickup_datetime            0
tpep_dropoff_datetime           0
passenger_count                 0
trip_distance                   0
RatecodeID                      0
store_and_fwd_flag              0
PULocationID                    0
DOLocationID                    0
payment_type                    0
fare_amount                     0
extra                           0
mta_tax                         0
tip_amount                      0
tolls_amount                    0
improvement_surcharge           3
total_amount                    0
congestion_surcharge     12741035
airport_fee              12741035
dtype: int64

In [100]:
# Boş sütunları kaldır
df = df.dropna(axis=1, how="all")

print("Remaining columns:", len(df.columns))

Remaining columns: 1


In [101]:
df.columns # 'congestion_surcharge' ve 'airport_fee' sütunları kaldırıldı

Index(['tpep_pickup_datetime'], dtype='str')

In [102]:
print("Dataset shape:", df.shape)

Dataset shape: (146039231, 1)


In [103]:
# pickup sütununu datetime türüne dönüştürülmesi
df["tpep_pickup_datetime"] = pd.to_datetime(
    df["tpep_pickup_datetime"]
)
# saatlik ytrip sayısı hesaplanması
january_hourly = (
    df_january.set_index("tpep_pickup_datetime")
    .resample("h")
    .size()
    .reset_index(name="trip_count")
)
january_hourly.head()

,tpep_pickup_datetime,trip_count
0,2015-01-01 00:00:00,28312
1,2015-01-01 01:00:00,31707
2,2015-01-01 02:00:00,28068
3,2015-01-01 03:00:00,24288
4,2015-01-01 04:00:00,17081


### yıl boyu trip sayısı hesaplanması

In [104]:
# Tüm aylık dosyalardan sadece pickup datetime sütununu okuyorum
all_data = []
for file in files:
    print("Reading:", file)

    temp_df = pd.read_parquet(
        file,
        columns=["tpep_pickup_datetime"]
    )
    all_data.append(temp_df)

# Tüm ayları tek dataframe içinde birleştiriyorum
df = pd.concat(all_data, ignore_index=True)
print("Full dataset shape:", df.shape)

Reading: ../data/yellow_tripdata_2015-10.parquet
Reading: ../data/yellow_tripdata_2015-09.parquet
Reading: ../data/yellow_tripdata_2015-08.parquet
Reading: ../data/yellow_tripdata_2015-11.parquet
Reading: ../data/yellow_tripdata_2015-01.parquet
Reading: ../data/yellow_tripdata_2015-03.parquet
Reading: ../data/yellow_tripdata_2015-12.parquet
Reading: ../data/yellow_tripdata_2015-02.parquet
Reading: ../data/yellow_tripdata_2015-07.parquet
Reading: ../data/yellow_tripdata_2015-06.parquet
Reading: ../data/yellow_tripdata_2015-04.parquet
Reading: ../data/yellow_tripdata_2015-05.parquet
Full dataset shape: (146039231, 1)


In [105]:
# tekrar datetime formatına dönüştürülmesi
df["tpep_pickup_datetime"] = pd.to_datetime(
    df["tpep_pickup_datetime"],
    errors="coerce"
)
# boş datetime satırlarını kaldırılması
df = df.dropna(
    subset=["tpep_pickup_datetime"]
)
# sadece 2015 yılı içindeki verileri tutuyorum
df = df[
    (df["tpep_pickup_datetime"] >= "2015-01-01") &
    (df["tpep_pickup_datetime"] < "2016-01-01")
]
df.head()

,tpep_pickup_datetime
0,2015-10-01 00:04:43
1,2015-10-01 00:13:58
2,2015-10-01 00:53:57
3,2015-10-01 00:59:04
4,2015-10-01 00:01:45


### Saatlik trip sayısı

In [106]:
# datetime sütununu index yapıp saatlik trip sayısını hesaplıyorum
hourly_data = (
    df.set_index("tpep_pickup_datetime")
    .resample("h")
    .size()
    .reset_index(name="trip_count")
)
hourly_data.head()

,tpep_pickup_datetime,trip_count
0,2015-01-01 00:00:00,28312
1,2015-01-01 01:00:00,31707
2,2015-01-01 02:00:00,28068
3,2015-01-01 03:00:00,24288
4,2015-01-01 04:00:00,17081


In [107]:
# saatlik veri boyutunu kontrol edilmesi
hourly_data.shape

(8760, 2)

In [108]:
# saatlik trip count verisini csv olarak kaydediyorum
hourly_data.to_csv(
    "../outputs/hourly_trip_counts_2015.csv",
    index=False
)
print("CSV dosyası kaydedildi")

CSV dosyası kaydedildi
